# 02 - Train CharCNN and select the September checkpoint

Attach two private Kaggle Datasets: `charcnn-project` (the updated code) and `deepurlbench-binary-temporal-splits` (the prepared data). Enable GPU T4. This notebook uses `train.parquet` only to fit weights and uses `validation.parquet` only for checkpoint selection and threshold calibration.

In [ ]:
import glob, os, shutil, sys

# Locate attached code and prepared data without hard-coding your Kaggle user name.
config_hits = glob.glob('/kaggle/input/**/configs/config.yaml', recursive=True)
train_hits = glob.glob('/kaggle/input/**/train.parquet', recursive=True)
assert config_hits, 'Updated charcnn-project Dataset is not attached.'
assert train_hits, 'deepurlbench-binary-temporal-splits Dataset is not attached.'
PROJECT_INPUT = os.path.dirname(os.path.dirname(config_hits[0]))
DATA_INPUT = os.path.dirname(train_hits[0])
WORKING_PROJECT = '/kaggle/working/charcnn'
shutil.copytree(PROJECT_INPUT, WORKING_PROJECT, dirs_exist_ok=True)
print('Code:', PROJECT_INPUT)
print('Prepared data:', DATA_INPUT)
%pip install -q -r {WORKING_PROJECT}/requirements.txt


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU accelerator: Settings -> Accelerator -> GPU T4 x2 or T4.'
print(torch.cuda.get_device_name(0))
os.chdir(WORKING_PROJECT)
sys.path.insert(0, f'{WORKING_PROJECT}/src')
from charcnn.train import main
# To resume, attach a prior artifacts Dataset and set this to its checkpoint path.
RESUME_CHECKPOINT = None  # e.g. '/kaggle/input/charcnn-training-artifacts/checkpoint_epoch_2.pt'
train_args = [
    '--config', f'{WORKING_PROJECT}/configs/config.yaml',
    '--data-dir', DATA_INPUT,
]
if RESUME_CHECKPOINT:
    train_args.extend(['--resume', RESUME_CHECKPOINT])
main(train_args)


In [ ]:
from pathlib import Path
artifacts = Path('/kaggle/working/charcnn/artifacts')
for path in sorted(artifacts.iterdir()):
    print(path.name, f'{path.stat().st_size / 1024**2:.1f} MB')
print((artifacts / 'validation_metrics.json').read_text())
